# Publish the fine-tuned LFM2 Extract: remaining export and model cards

Starts from the merged model already on the Hub (no training), builds the int8 CPU export that is still missing from the ONNX repo, and writes a model card and licence into both repos.

Why int8: the WebGPU `q4f16` export copies sections faithfully, but the CPU `q4` export of this checkpoint paraphrases instead of copying (the stock model's CPU export does not, so it is these weights under the fp32 int4 graph). An int8 CPU file is the candidate replacement for running on the CPU: in Node (`run.mjs --dtype int8`) and the browser's wasm backend. The check cell below says whether it copies.

A CPU runtime is enough; the builder does not use a GPU. Fill in the first cell and run top to bottom.

In [ ]:
HF_USER = "playforgecoding"
BASE_MODEL = "LiquidAI/LFM2-1.2B-Extract"
MERGED_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson"
ONNX_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson-ONNX"

# What the training run and its held-out check produced, for the model card.
# Edit if the numbers differ for your run.
TRAINING = dict(
    lessons=13, layouts=7, train_examples=412, holdout_examples=84,
    epochs=3, lora_rank=16, learning_rate="2e-4", max_length=4096,
    holdout_sample=20, parsed="95%", spelling="85%", prompts="95%", answers="98%", invented=1,
)
REPO_URL = "https://github.com/Spelling-Creator/spelling-creator"
DOCS_URL = "https://spellingcreator.org/docs/monorepo/document-import-experiment"
RELAYOUT_URL = f"https://raw.githubusercontent.com/Spelling-Creator/spelling-creator/main/packages/core/scripts/extract-eval/relayout-onnx.py"

In [ ]:
%pip install -q onnxruntime-genai onnx onnx_ir onnxscript huggingface_hub

## Build the int8 CPU export

In [ ]:
import os, subprocess, sys, urllib.request
from huggingface_hub import snapshot_download, hf_hub_download

snapshot_download(MERGED_REPO, local_dir="merged")
subprocess.run(
    [sys.executable, "-m", "onnxruntime_genai.models.builder",
     "-i", "merged", "-o", "build-cpu-int8", "-p", "int8", "-e", "cpu", "-c", "hf-cache"],
    check=True,
)

# The relayout script from the repo. Upload it beside the notebook with the
# file panel, or let this fetch it from main once the branch carrying it has merged.
if not os.path.exists("relayout-onnx.py"):
    urllib.request.urlretrieve(RELAYOUT_URL, "relayout-onnx.py")

# Start the destination from the ONNX repo's current config.json, so the
# relayout merges the int8 entry into the existing q4 and q4f16 entries rather
# than replacing them.
os.makedirs("onnx-repo", exist_ok=True)
hf_hub_download(ONNX_REPO, "config.json", local_dir="onnx-repo")
subprocess.run([sys.executable, "relayout-onnx.py", "build-cpu-int8", "onnx-repo", "int8", "merged"], check=True)
print(os.listdir("onnx-repo"), os.listdir("onnx-repo/onnx"))

## Check that the int8 export copies rather than paraphrases

Runs the builder's own output through onnxruntime-genai on one section from the held-out lesson. The reply should start with the section's first paragraph copied word for word ("Every pet cat curled up on a sofa today..."), not a summary of it.

In [ ]:
import json
import onnxruntime_genai as og

SCHEMA = json.dumps({
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "The section heading, or an empty string if there is none."},
        "paragraphs": {"type": "array", "description": "The passage: one item per paragraph, each paragraph copied in full, word for word, starting with the very first paragraph. Paragraphs are separated by blank lines in the document; a paragraph is a whole block of several sentences, never a single sentence.", "items": {"type": "string"}},
        "spellingWords": {"type": "array", "description": "The words on the spelling line, in order, exactly as written.", "items": {"type": "string"}},
        "questions": {"type": "array", "description": "Every question, in order.", "items": {"type": "object", "properties": {
            "prompt": {"type": "string", "description": "The question as written, without its answer."},
            "type": {"type": "string", "enum": ["single", "number", "multiple", "multiple_open", "background", "open", "wyr", "paraphrase"]},
            "answers": {"type": "array", "description": "The answer or answers printed with the question, exactly as written. Empty when none is printed.", "items": {"type": "string"}},
            "steps": {"type": "array", "description": "Numbered working-out lines under a number question. Usually empty.", "items": {"type": "string"}}},
            "required": ["prompt", "type", "answers"]}},
    },
    "required": ["name", "paragraphs", "spellingWords", "questions"],
}, indent=1)

SECTION = """The Wildcat Ancestor

Every pet cat curled up on a sofa today is descended from a single wild species: the AFRICAN WILDCAT, known to scientists as Felis lybica. This sandy-coloured hunter still roams the DESERTS, GRASSLANDS, and PLAINS of North Africa and the Near East. It looks much like a large, long-legged tabby, and it lives alone, hunting by night.

The story of how this shy WILDCAT became a house cat began with farming. When people first stored harvests of grain, the great piles drew MICE, BIRDS, and INSECTS in enormous numbers. The wildcats followed that easy prey into human VILLAGES. Rather than being captured, the boldest cats simply chose to stay, and this slow, self-guided change is called DOMESTICATION.

Spelling words: FELINE, CREATURE, PROWLED, NATURAL

1. From which single wild species is every pet cat descended? (Answer: THE AFRICAN WILDCAT)
2. What is the name for the slow change of a wild animal into a tame one? (Answer: DOMESTICATION)
3. Name a wild animal that hunts.
4. Would you rather hunt by night or by day?
"""

model = og.Model("build-cpu-int8")
tokenizer = og.Tokenizer(model)
prompt = (
    "<|startoftext|><|im_start|>system\nReturn data as a JSON object with the following schema:\n" + SCHEMA +
    "<|im_end|>\n<|im_start|>user\n" + SECTION + "<|im_end|>\n<|im_start|>assistant\n"
)
params = og.GeneratorParams(model)
params.set_search_options(max_length=2500, do_sample=False)
generator = og.Generator(model, params)
generator.append_tokens(tokenizer.encode(prompt))
out = []
while not generator.is_done() and len(out) < 700:
    generator.generate_next_token()
    out.append(generator.get_next_tokens()[0])
reply = tokenizer.decode(out)
print(reply[:1200])
print("\ncopies faithfully:", "Every pet cat curled up on a sofa today is descended" in reply)

If the last line says `False`, do not upload the int8 file: edit the upload cell to skip the `onnx/` folder, and push only the model cards. Otherwise run on.

## Model cards and licence

Both repos derive from LFM2-1.2B-Extract, released under the LFM Open License v1.0, which the derived repos carry too.

In [ ]:
hf_hub_download(BASE_MODEL, "LICENSE", local_dir="base")
T = TRAINING

FRONT = f"""---
license: other
license_name: lfm1.0
license_link: LICENSE
language:
- en
pipeline_tag: text-generation
tags:
- liquid
- lfm2
- edge
- extraction
- spelling-creator
"""

WHAT = f"""## What it does

Turns one section of a spelling lesson document, as a person might type it up or as this app's own Word export reads back as plain text, into the lesson's JSON: the passage paragraphs copied word for word, the spelling words, and every question with its printed answers and working-out. It is a fine-tune of [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}) for [Spelling Creator]({REPO_URL}), a lesson builder for Spelling to Communicate (S2C), where lessons are read aloud to nonspeaking spellers.

The stock Extract model handled this badly: it split paragraphs into sentences, invented answers for open questions, and often abandoned the schema. This fine-tune copies rather than invents.

## Training

LoRA (rank {T['lora_rank']}, learning rate {T['learning_rate']}, {T['epochs']} epochs, {T['max_length']}-token context) on {T['train_examples']} examples: the {T['lessons']} published lessons on the Spelling Creator hub, each rendered in {T['layouts']} document layouts (the app's Word export read as raw text, and six typed-up styles) and cut into sections, paired with the section's lesson JSON. {T['holdout_examples']} sections from the two newest lessons were held out. The data generator and the training notebook are in the repo under `packages/core/scripts/extract-eval/`.

## Results

On a sample of {T['holdout_sample']} held-out sections spread across both held-out lessons and all layouts:

| metric | value |
| --- | ---: |
| sections parsed as JSON | {T['parsed']} |
| spelling words exact | {T['spelling']} |
| question prompts found | {T['prompts']} |
| answers exact | {T['answers']} |
| answers invented for open questions | {T['invented']} |

For comparison, the stock model's answers were right about half the time on a tidy typed document and a quarter of the time on the Word export. Question types are not reliable from this or any small model; Spelling Creator derives the type from the answers and wording instead. The rule-based parser in the app still beats this model on every regular layout; the model is for documents the parser cannot read. The full write-up: [{DOCS_URL}]({DOCS_URL}).

## Prompt

The system prompt is `Return data as a JSON object with the following schema:` followed by the JSON Schema of a section (`name`, `paragraphs`, `spellingWords`, `questions[{{prompt, type, answers, steps}}]`), and the user turn is the section's text. Greedy decoding.
"""

MERGED_CARD = FRONT + f"""library_name: transformers
base_model: {BASE_MODEL}
---

# LFM2-1.2B-Extract, fine-tuned for Spelling Creator lesson documents

The merged weights (bf16, safetensors). For the browser or Node, use the ONNX export at [{ONNX_REPO}](https://huggingface.co/{ONNX_REPO}).

{WHAT}
## Use

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("{MERGED_REPO}")
model = AutoModelForCausalLM.from_pretrained("{MERGED_REPO}", dtype="bfloat16", device_map="auto")
messages = [
    {{"role": "system", "content": "Return data as a JSON object with the following schema:\\n" + SCHEMA}},
    {{"role": "user", "content": section_text}},
]
text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
out = model.generate(ids, max_new_tokens=1500, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
```

## Licence

Derived from LFM2-1.2B-Extract and released under the same LFM Open License v1.0 (see LICENSE).
"""

ONNX_CARD = FRONT + f"""library_name: transformers.js
base_model: {MERGED_REPO}
---

# LFM2-1.2B-Extract, fine-tuned for Spelling Creator lesson documents (ONNX)

The ONNX export of [{MERGED_REPO}](https://huggingface.co/{MERGED_REPO}) for [Transformers.js](https://huggingface.co/docs/transformers.js), made with the onnxruntime-genai model builder and laid out the way the onnx-community LFM2 files are.

{WHAT}
## Files

| file | for | note |
| --- | --- | --- |
| `onnx/model_q4f16.onnx` | WebGPU (`device: "webgpu"`, `dtype: "q4f16"`) | Verified: copies a real section faithfully in Chromium. The one to ship in a browser. |
| `onnx/model_int8.onnx` | CPU: Node and the wasm backend (`dtype: "int8"`) | See the model's own card for whether this build passed the copy check. |
| `onnx/model_q4.onnx` | CPU | Not faithful for this checkpoint: paraphrases instead of copying. Kept for reference. |

## Use

```javascript
import {{ AutoModelForCausalLM, AutoTokenizer }} from "@huggingface/transformers";

const id = "{ONNX_REPO}";
const tokenizer = await AutoTokenizer.from_pretrained(id);
const model = await AutoModelForCausalLM.from_pretrained(id, {{ dtype: "q4f16", device: "webgpu" }});
const inputs = tokenizer.apply_chat_template(
  [
    {{ role: "system", content: "Return data as a JSON object with the following schema:\\n" + SCHEMA }},
    {{ role: "user", content: sectionText }},
  ],
  {{ add_generation_prompt: true, return_dict: true }},
);
const output = await model.generate({{ ...inputs, max_new_tokens: 1500, do_sample: false }});
console.log(tokenizer.batch_decode(output.slice(null, [inputs.input_ids.dims[1], null]), {{ skip_special_tokens: true }})[0]);
```

## How it was converted

`python -m onnxruntime_genai.models.builder -i merged -p int4 -e webgpu` (q4f16) and `-e cpu` builds, then `relayout-onnx.py` from the Spelling Creator repo: it renames the convolution caches to `past_conv.N`, makes the KV cache's head dimension concrete, moves the chat template into `tokenizer_config.json`, adds the `transformers.js_config` block, and puts the graphs under `onnx/`.

## Licence

Derived from LFM2-1.2B-Extract and released under the same LFM Open License v1.0 (see LICENSE).
"""

import shutil
os.makedirs("merged-card", exist_ok=True)
open("merged-card/README.md", "w").write(MERGED_CARD)
shutil.copy("base/LICENSE", "merged-card/LICENSE")
open("onnx-repo/README.md", "w").write(ONNX_CARD)
shutil.copy("base/LICENSE", "onnx-repo/LICENSE")
print(MERGED_CARD[:600])

## Upload

In [ ]:
from huggingface_hub import login, HfApi

login()
api = HfApi()
api.upload_folder(folder_path="merged-card", repo_id=MERGED_REPO, commit_message="Model card and licence")
api.upload_folder(folder_path="onnx-repo", repo_id=ONNX_REPO, commit_message="int8 CPU export, model card and licence")
print("pushed", MERGED_REPO, "and", ONNX_REPO)